# 01. load raw data

## 0. setup

In [1]:
import gc
from pathlib import Path
 
import numpy as np
import pandas as pd

# paths
root         = Path.cwd().parent
data_raw     = root / 'data' / 'raw'
data_interim = root / 'data' / 'interim'
data_proc    = root / 'data' / 'processed'

data_interim.mkdir(parents=True, exist_ok=True)

## 1. config

In [2]:
# sample bound
year_max = 2024

# patent inputs
pat_raw = data_raw / 'patent'
pat_files = {
    'epo_app': pat_raw / '202602_EPO_App_reg.txt',
    'pct_app': pat_raw / '202602_PCT_App_reg.txt',
    'epo_ipc': pat_raw / '202602_EPO_IPC.txt',
    'pct_ipc': pat_raw / '202602_PCT_IPC.txt',
    'epo_pct': pat_raw / '202602_EPO_PCT.txt',
    'tpf_epo': pat_raw / '202602_TPF_EPO.txt',
    'tpf_pct': pat_raw / '202602_TPF_PCT.txt',
    'epo_cit': pat_raw / '202602_EPO_CIT_COUNTS.txt'
}

# output
out_pat = data_interim / 'pat_data.parquet'
print(f'inputs : {pat_raw.name}/  ->  output: {out_pat.name}')

inputs : patent/  ->  output: pat_data.parquet


## 2. patents

### 2.1 load epo and pct

In [3]:
# applicants
epo_app = pd.read_csv(
    pat_files['epo_app'], sep='|',
    usecols=['appln_id', 'app_nbr', 'person_id', 'ctry_code', 'app_share']
).rename(columns={'app_nbr': 'pat_nbr'})

pct_app = pd.read_csv(
    pat_files['pct_app'], sep='|',
    usecols=['pct_nbr', 'appln_id', 'app_name', 'ctry_code', 'app_share']
).rename(columns={'pct_nbr': 'pat_nbr'})

# ipc codes
epo_ipc = pd.read_csv(pat_files['epo_ipc'], sep='|', low_memory=False).rename(columns={'IPC': 'ipc'})
pct_ipc = pd.read_csv(pat_files['pct_ipc'], sep='|', low_memory=False).rename(columns={'IPC': 'ipc', 'pct_nbr': 'pat_nbr'})

# epo-pct correspondence
epo_pct = pd.read_csv(pat_files['epo_pct'], sep='|')

print(f'epo_app: {len(epo_app):,} | pct_app: {len(pct_app):,}')
print(f'epo_ipc: {len(epo_ipc):,} | pct_ipc: {len(pct_ipc):,}')

epo_app: 4,963,719 | pct_app: 5,713,312
epo_ipc: 19,247,975 | pct_ipc: 19,008,663


In [4]:
def _dedup(df, name):
    n0 = len(df)
    out = df.drop_duplicates()
    print(f'{name}: {n0:,} -> {len(out):,} (dropped {n0 - len(out):,}, {(n0 - len(out)) / n0:.1%})')
    return out

epo_app = _dedup(epo_app, 'epo_app')
pct_app['app_name'] = pct_app['app_name'].astype(str).str.upper().str.strip()
pct_app = _dedup(pct_app, 'pct_app')
epo_ipc = _dedup(epo_ipc, 'epo_ipc')
pct_ipc = _dedup(pct_ipc, 'pct_ipc')

epo_app: 4,963,719 -> 4,958,521 (dropped 5,198, 0.1%)
pct_app: 5,713,312 -> 5,706,899 (dropped 6,413, 0.1%)
epo_ipc: 19,247,975 -> 19,247,975 (dropped 0, 0.0%)
pct_ipc: 19,008,663 -> 19,008,464 (dropped 199, 0.0%)


### 2.2 merge applicants × ipc

In [5]:
ep = epo_app.merge(epo_ipc, on='appln_id', how='inner')
ep['publn_auth'] = 'ep'
ep['applt_id'] = ep['person_id'].astype('Int64').astype(str)
ep = ep.drop(columns='person_id')

wo = pct_app.merge(pct_ipc, on='pat_nbr', how='inner')
wo['publn_auth'] = 'wo'
wo['applt_id'] = wo['app_name']
wo = wo.drop(columns='app_name')

del epo_app, epo_ipc, pct_app, pct_ipc
gc.collect()

36

### 2.3 deduplicate dual filings (keep earliest; ties to EPO)

In [6]:
ep_years = ep.loc[ep['app_year'] <= year_max].groupby('pat_nbr')['app_year'].min().rename('ep_year')
wo_years = wo.loc[wo['app_year'] <= year_max].groupby('pat_nbr')['app_year'].min().rename('wo_year')

epo_pct = epo_pct.assign(app_nbr=epo_pct['app_nbr'].astype(str), pct_nbr=epo_pct['pct_nbr'].astype(str))
ep_years.index = ep_years.index.astype(str)
wo_years.index = wo_years.index.astype(str)

corr = (epo_pct
        .merge(ep_years, left_on='app_nbr', right_index=True, how='inner')
        .merge(wo_years, left_on='pct_nbr', right_index=True, how='inner'))
assert len(corr), 'no EPO-PCT correspondences matched - check file'

ep_to_drop = set(corr.loc[corr['ep_year'] >  corr['wo_year'], 'app_nbr'])
wo_to_drop = set(corr.loc[corr['wo_year'] >= corr['ep_year'], 'pct_nbr'])

n_ep0, n_wo0 = len(ep), len(wo)
ep = ep[~ep['pat_nbr'].astype(str).isin(ep_to_drop)].copy()
wo = wo[~wo['pat_nbr'].astype(str).isin(wo_to_drop)].copy()

del ep_years, wo_years, corr, ep_to_drop, wo_to_drop, epo_pct
gc.collect()
print(f'ep/wo dedup: ep {n_ep0:,} -> {len(ep):,} | wo {n_wo0:,} -> {len(wo):,}')

ep/wo dedup: ep 20,884,090 -> 20,873,936 | wo 21,092,566 -> 9,034,592


### 2.4 concat + drop keys + clip years

In [7]:
pat_df = pd.concat([ep, wo], ignore_index=True)
del ep, wo; gc.collect()

n0 = len(pat_df)
pat_df = pat_df.dropna(subset=['appln_id', 'applt_id', 'app_year', 'ipc', 'ctry_code', 'app_share'])
print(f'dropna on key cols: {n0:,} -> {len(pat_df):,}')

pat_df['app_year'] = pat_df['app_year'].astype('Int64')
pat_df = pat_df[pat_df['app_year'] <= year_max].copy()
print(f'raw merged: {pat_df['pat_nbr'].nunique():,} patents | {len(pat_df):,} rows (patent x applicant x ipc)')
print(pat_df.groupby('publn_auth')['pat_nbr'].nunique())

dropna on key cols: 29,908,528 -> 29,898,656
raw merged: 7,260,586 patents | 29,828,670 rows (patent x applicant x ipc)
publn_auth
ep    4587253
wo    2673333
Name: pat_nbr, dtype: int64


### 2.5 deduplicate triadic patent families (keep earliest application)

In [8]:
tpf_pat = pd.concat([
    pd.read_csv(pat_files['tpf_epo'], sep='|', usecols=['Family_id', 'Appln_id']).rename(columns=str.lower),
    pd.read_csv(pat_files['tpf_pct'], sep='|', usecols=['Family_id', 'Appln_id']).rename(columns=str.lower),
], ignore_index=True).drop_duplicates(subset=['appln_id'])

pat_df = pat_df.merge(tpf_pat, on='appln_id', how='left')

in_tpf = pat_df['family_id'].notna()
keep = (pat_df.loc[in_tpf, ['family_id', 'appln_id', 'app_year']]
        .drop_duplicates(subset=['family_id', 'appln_id'])
        .sort_values(['family_id', 'app_year', 'appln_id'])
        .drop_duplicates('family_id', keep='first')[['family_id', 'appln_id']])
keep_set = set(keep['appln_id'])

pat_df = pat_df[(~in_tpf) | (pat_df['appln_id'].isin(keep_set))].copy()
pat_df = pat_df.drop(columns='family_id')
del keep, keep_set, tpf_pat; gc.collect()

print(f'after tpf dedup: {pat_df['appln_id'].nunique():,} patents | {len(pat_df):,} rows | '
      f'{pat_df['app_year'].min()}-{pat_df['app_year'].max()}')

after tpf dedup: 6,880,084 patents | 27,384,692 rows | 1978-2024


### 2.6 forward citations

In [9]:
# forward-citation counts per EP patent
cit = pd.read_csv(pat_files['epo_cit'], sep='|',
                  usecols=['EP_Appln_id', 'WO_Appln_id',
                           'Direct_cits_Recd', 'Direct_cits_Recd_in3'])

ep_key = (cit[['EP_Appln_id', 'Direct_cits_Recd', 'Direct_cits_Recd_in3']]
          .rename(columns={'EP_Appln_id': 'appln_id'}))
ep_key['_src'] = 0                                    # EP wins any key collision
wo_key = (cit.loc[cit['WO_Appln_id'].notna(),
                  ['WO_Appln_id', 'Direct_cits_Recd', 'Direct_cits_Recd_in3']]
          .rename(columns={'WO_Appln_id': 'appln_id'}))
wo_key['appln_id'] = wo_key['appln_id'].astype('int64')
wo_key['_src'] = 1

cit_key = (pd.concat([ep_key, wo_key], ignore_index=True)
           .sort_values('_src')
           .drop_duplicates('appln_id', keep='first')  # deterministic: EP over WO
           .drop(columns='_src')
           .rename(columns={'Direct_cits_Recd':     'cit_recd',      # lifetime, EP/WO direct
                            'Direct_cits_Recd_in3': 'cit_recd_3yr'})) # 3-yr window (headline)

# patent merge
pat_df['appln_id'] = pat_df['appln_id'].astype('int64')
cit_key['appln_id'] = cit_key['appln_id'].astype('int64')
pat_df = pat_df.merge(cit_key, on='appln_id', how='left')

# unmatched = outside the EP citation frame assign NaN + flag, NEVER filled 0
pat_df['cit_matched'] = pat_df['cit_recd'].notna()

del cit, ep_key, wo_key, cit_key; gc.collect()

m = pat_df.drop_duplicates('appln_id')
print(f'citation match: {m["cit_matched"].mean():.1%} of patents '
      f'({int(m["cit_matched"].sum()):,}/{len(m):,})')
print('by authority:')
print(m.groupby('publn_auth')['cit_matched'].mean().round(3).to_string())
print('cit_recd_3yr (matched):',
      m.loc[m['cit_matched'], 'cit_recd_3yr'].describe(percentiles=[.5,.9]).round(2).to_dict())

citation match: 61.9% of patents (4,261,333/6,880,084)
by authority:
publn_auth
ep    1.000
wo    0.001
cit_recd_3yr (matched): {'count': 4261333.0, 'mean': 0.48, 'std': 1.24, 'min': 0.0, '50%': 0.0, '90%': 2.0, 'max': 185.0}


### 2.7 diagnostics

In [10]:
print('patents')
print(f'  rows         : {len(pat_df):,}')
print(f'  applications : {pat_df['appln_id'].nunique():,}')
print(f'  applicants   : {pat_df['applt_id'].nunique():,}')
print(f'  cit-matched  : {pat_df.drop_duplicates("appln_id")["cit_matched"].mean():.1%}')
print(f'  years        : {pat_df['app_year'].min()}-{pat_df['app_year'].max()}')
print(f'  countries    : {pat_df['ctry_code'].nunique()}')

patents
  rows         : 27,384,692
  applications : 6,880,084
  applicants   : 1,518,934
  cit-matched  : 61.9%
  years        : 1978-2024
  countries    : 232


### 2.7 save

In [11]:
pat_df.to_parquet(out_pat, index=False)
print(f'saved: {out_pat.name} ({len(pat_df):,} rows)')

del pat_df; gc.collect()

saved: pat_data.parquet (27,384,692 rows)


0